In [2]:
from ultralytics import YOLO
import cv2

# Load YOLOv8 Model
model = YOLO("yolov8n.pt")   

# Input & Output Video
input_video = "people.mp4"         
output_video = "people_counted.mp4"

cap = cv2.VideoCapture(input_video)

# Get video properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# Process Video
while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    # Track objects
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    people_count = 0

    if results[0].boxes is not None:
        boxes = results[0].boxes

        for box in boxes:
            if int(box.cls[0]) == 0:
                people_count += 1
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = float(box.conf[0])
               
                if box.id is not None:
                    track_id = int(box.id[0])
                else:
                    track_id = -1

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

                # Label
                label = f"ID:{track_id} {confidence:.2f}"

                cv2.putText(
                    frame,
                    label,
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,255,0),
                    2
                )

    # Live People Count
    cv2.putText(
        frame,
        f"People Count: {people_count}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        3
    )

    # Show frame
    cv2.imshow("People Counting", frame)

    # Save frame
    out.write(frame)

    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Processing Completed!")
print("Output saved as:", output_video)

Processing Completed!
Output saved as: people_counted.mp4
